# ContractRisk — integración OCR 20

OCR 19 leyó 8/8 imágenes completas de su muestra; eso no comprobó las respuestas contractuales. Este notebook verifica un **perfil nuevo y separado**, con OCR en CPU, generador FP16, recuperación, herramientas y M1. No carga el VLM Qwen; su línea base y sus resultados permanecen intactos. No se heredan sus notas ni sus métricas de eficiencia.

Una pasada: los **22 casos comunes originales**, sin ampliarlos ni cambiar referencias, y **16 comprobaciones separadas** de integración. Solo pregunta/imagen/descripcion entran al sistema; referencias se usan después. Los diagnósticos automáticos no son revisión humana ni notas del juez.

Requiere **GPU T4** para el generador. No requiere secretos ni API pagada. No cambia main ni el Space. Si falla una celda, pasar a la última para conservar el parcial; no repetir buscando respuestas mejores.

In [1]:
import os,sys,subprocess,tempfile,json
from pathlib import Path
os.environ['USE_TF']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['ANONYMIZED_TELEMETRY']='false'
SETUP_OK=False
ROOT=Path(tempfile.mkdtemp(prefix='contractrisk_integration20_',dir='/content'))/'repo'
subprocess.run(['git','clone','--single-branch','--branch','Entrega/contractrisk-final','https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',str(ROOT)],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Código:',COMMIT)
def run_visible(arguments,logfile):
    with open(logfile,'w',encoding='utf-8') as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError('Proceso detenido; conservar parcial con la última celda: '+logfile)


Código: 1b0e080daa21b9d3298a58f6217c13c55eb44e97


In [2]:
SETUP_OK=False
run_visible([sys.executable,'final/prepare_test_history.py'],'integration20_history.log')
run_visible([sys.executable,'-m','pip','install','-r','final/space_candidate/requirements.txt'],'integration20_install.log')
run_visible([sys.executable,'final/service_environment.py','--output','integration20_environment.json'],'integration20_environment.log')
run_visible(['apt-get','update'],'integration20_apt.log')
run_visible(['apt-get','install','-y','tesseract-ocr','tesseract-ocr-spa','tesseract-ocr-eng'],'integration20_ocr_install.log')
run_visible(['tesseract','--version'],'integration20_ocr_version.log')
run_visible([sys.executable,'-c',"import torch; assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Selecciona GPU T4'; print(torch.cuda.get_device_name(0))"],'integration20_gpu.log')
run_visible([sys.executable,'-m','unittest','discover','-s','final/tests','-v'],'integration20_tests_final.log')
run_visible([sys.executable,'-m','unittest','discover','-s','m3/tests','-v'],'integration20_tests_m3.log')
SETUP_OK=True


{
  "source_commit": "1b0e080daa21b9d3298a58f6217c13c55eb44e97",
  "was_shallow": false,
  "history_complete": true,
  "checkout_unchanged": true,
  "historical_commits_verified": [
    "6ec8dcdf22433c632b5eb0e5539401c33b9c923d",
    "71912f1c5755790381a8e39c8a67801bcbf9243d",
    "7afdda0212bf38ad1accd043a968db722569c618",
    "ece58275c2a19abfb6bbd2fe4b507731232e9083"
  ]
}
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 41.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 80.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.7

In [3]:
if not SETUP_OK:raise RuntimeError('Preparación incompleta; descargar diagnóstico sin ejecutar modelos.')
RESULTS=ROOT/'integration20_results'
run_visible([sys.executable,'-u','final/run_integration20.py','--output',str(RESULTS)],'integration20_inference.log')


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given

Fetching 7 files: 100%|██████████| 7/7 [00:10<00:00,  1.53s/it]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at BSC-LT/RoBERTalex and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 2 files: 100%|██████████| 2/2 [00:52<00:00, 26.24s/it]

Loadi

## Lectura y evidencia

`completed_semantic_review_pending` indica que terminó la recolección, no que se aprobó calidad. `common_predictions.jsonl` conserva los 22 casos del perfil nuevo; `integration_checks.jsonl` conserva 16 pruebas de fuentes/guardas/M1. Igualdad de guardas, campos o cifras no es evaluación semántica completa. No presentar sus diagnósticos como dominio humano o nota de juez.

No se ejecutó una comparación de eficiencia en esta prueba. Cambiar el lector visual modifica el sistema: no se le atribuye el 16/22 previo de Qwen ni se suma esta muestra a la tabla común. No subir archivos al Space hasta revisar.

In [4]:
import zipfile
from google.colab import files
metadata=ROOT/'integration20_results/metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'La inferencia no comenzó o no creó metadata; revisar logs.')
archive=ROOT.parent/'contractrisk_final_integracion20_resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    base=ROOT/'integration20_results'
    if base.exists():
        for item in sorted(base.rglob('*')):
            if item.is_file():z.write(item,item.relative_to(ROOT))
    for pattern in ['integration20_*.log','integration20_environment.json']:
        for item in sorted(ROOT.glob(pattern)):z.write(item,item.name)
    z.writestr('checkout_commit.txt',COMMIT+'\n')
    z.writestr('setup_state.json',json.dumps({'setup_ok':SETUP_OK}))
files.download(str(archive))


{
  "status": "completed_semantic_review_pending",
  "version": "integrated-ocr-20",
  "profile": "integrated_ocr20_fp16",
  "judge_used": false,
  "api_calls": 0,
  "semantic_quality_approved": false,
  "public_deployment_approved": false,
  "source_commit": "1b0e080daa21b9d3298a58f6217c13c55eb44e97",
  "started_utc": "2026-11-20T01:10:14.604135+00:00",
  "common_completed": 22,
  "checks_completed": 16,
  "common_inputs_sha256": "da677d46731d5205a4b664b0c3f3155c138fac9b799115d53e753a1204b13dd6",
  "common_references_sha256": "6103fb2177c1f3d17efdaec7958549e96f179731dfd49bc16c986cd53df00f93",
  "active_neural_models": [
    "generator",
    "embedding",
    "reranker",
    "classifier"
  ],
  "qwen_vlm_loaded": false,
  "efficiency_comparison_approved": false,
  "model_revisions": {
    "embedding": [
      "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
      "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
    ],
    "generator": [
      "Qwen/Qwen2.5-3B-Instruct",
      "a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>